# Bayesian Walk (Posterior Sampling) - Fine-Tuning

Like Bayesian Walk, but uses posterior marginals instead of softmax over expected values.
- Stage 1: role dist = posterior marginal
- Stage 2+: stick with prob (1 - eps_switch), or switch via posterior marginal with prob eps_switch

Params: tau_prior, epsilon, eps_switch (3 params, no tau_softmax or values needed)

In [1]:
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from itertools import product
from scipy.optimize import minimize

# Shared package
from shared import EXPORTS_DIR, DATA_ROOT
from shared.constants import (
    F, T, M,
    ROLE_NAMES, ROLE_SHORT, ROLE_CHAR_TO_IDX, GAME_ROLE_TO_IDX,
    ALL_ROLE_COMBOS, EPSILON, MAX_STAGES, TURNS_PER_STAGE,
)
from shared.parsing import canonical_combo, get_canonical_combos
from shared.inference import (
    utility_based_prior, uniform_prior,
    bayesian_update, action_prob, preferred_action, game_step,
    combo_marginal,
    ATTACK, DEFEND, HEAL,
)
from shared.evaluation import (
    run_predictions, compute_pearson, compute_log_likelihood, extract_metrics,
)

# Still need online_model_sim for load_team_rounds
OMS_DIR = Path(DATA_ROOT).parent.parent / 'computational_model' / 'analysis'
sys.path.insert(0, str(OMS_DIR))
import online_model_sim as oms

In [2]:
# Monkey-patch env paths to use shared data directory
oms.VALUE_MATRICES_DIR = DATA_ROOT / 'human_envs_value_matrices'
oms.ENVS_DIR = DATA_ROOT / 'envs'

# Monkey-patch config loader to avoid JAX dependency
from shared.env_loading import load_env_config
import re as _re

def _load_config_no_jax(config_path):
    text = Path(config_path).read_text()
    team_max_hp = int(_re.search(r'TEAM_MAX_HP\s*=\s*(\d+)', text).group(1))
    enemy_max_hp = int(_re.search(r'ENEMY_MAX_HP\s*=\s*(\d+)', text).group(1))
    boss_damage = float(_re.search(r'BOSS_DAMAGE\s*=\s*([\d.]+)', text).group(1))
    ps_match = _re.search(r'PLAYER_STATS\s*=\s*(?:jnp\.array|np\.array)?\(?\s*(\[\[.+?\]\])\s*\)?', text, _re.DOTALL)
    rows = _re.findall(r'\[([^\[\]]+)\]', ps_match.group(1))
    player_stats = np.array([[float(x) for x in row.split(',')] for row in rows])
    class Config: pass
    cfg = Config()
    cfg.TEAM_MAX_HP = team_max_hp
    cfg.ENEMY_MAX_HP = enemy_max_hp
    cfg.BOSS_DAMAGE = boss_damage
    cfg.PLAYER_STATS = player_stats
    return cfg

oms.load_config_module = _load_config_no_jax

# Load data
DATA_DIRS = [
    str(EXPORTS_DIR / 'bayesian-role-specialization-2026-03-06-09-54-19'),
    str(EXPORTS_DIR / 'bayesian-role-specialization-2026-03-18-15-47-09'),
]

records = oms.load_team_rounds(data_dirs=DATA_DIRS)
n_envs = len(set(r['env_id'] for r in records))
print(f'Loaded {len(records)} team-rounds across {n_envs} environments')

Loaded 66 team-rounds across 6 environments


## Model Definition

In [3]:
def posterior_marginal(prior, agent_i):
    """Marginalize the (3,3,3) joint posterior to get P(role) for agent_i."""
    marg = np.sum(prior, axis=tuple(j for j in range(3) if j != agent_i))
    total = marg.sum()
    return marg / total if total > 0 else np.ones(3) / 3.0


def make_bayesian_walk_ps(tau_prior=2.0, epsilon=0.2, epsilon_switch=0.3):
    """Bayesian Walk with posterior sampling: stick-or-switch using posterior marginals."""
    def predict_fn(record):
        env = record['env_config']
        player_stats = env['player_stats']
        boss_damage = env['boss_damage']
        team_max_hp, enemy_max_hp = env['team_max_hp'], env['enemy_max_hp']

        team_hp = float(team_max_hp)
        enemy_hp = float(enemy_max_hp)
        prior = utility_based_prior(player_stats, tau=tau_prior)
        results = []
        turn_idx = 0
        prev_roles = None

        for human_combo in record['stage_roles']:
            if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                break

            # Posterior marginal for each agent
            switch_dist = [posterior_marginal(prior, i) for i in range(3)]

            per_agent = []
            for i in range(3):
                if prev_roles is None:
                    per_agent.append(switch_dist[i])
                else:
                    stick = np.zeros(3)
                    stick[prev_roles[i]] = 1.0
                    mixed = (1.0 - epsilon_switch) * stick + epsilon_switch * switch_dist[i]
                    per_agent.append(mixed)

            predicted_dist = {}
            for r0 in range(3):
                for r1 in range(3):
                    for r2 in range(3):
                        combo = ROLE_SHORT[r0] + ROLE_SHORT[r1] + ROLE_SHORT[r2]
                        predicted_dist[combo] = float(per_agent[0][r0] * per_agent[1][r1] * per_agent[2][r2])

            results.append({
                'predicted_dist': predicted_dist,
                'human_combo': human_combo,
                'model_marginal': np.mean(per_agent, axis=0),
            })

            human_roles = [ROLE_CHAR_TO_IDX[c] for c in human_combo]
            prev_roles = human_roles
            for _ in range(TURNS_PER_STAGE):
                if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                    break
                intent = record['lds'][turn_idx]
                actions = [preferred_action(human_roles[i], intent, team_hp, team_max_hp)
                           for i in range(3)]
                prior = bayesian_update(prior, actions, intent, team_hp, team_max_hp, epsilon)
                team_hp, enemy_hp = game_step(intent, team_hp, enemy_hp, actions,
                                              player_stats, boss_damage, team_max_hp)
                turn_idx += 1

        return results
    return predict_fn


def precompute_trajectories(records, tau_prior, epsilon):
    """Precompute posterior + game state per stage for each record.

    Depends only on (tau_prior, epsilon) and observed human actions.
    """
    trajectories = []
    for record in records:
        env = record['env_config']
        player_stats = env['player_stats']
        boss_damage = env['boss_damage']
        team_max_hp, enemy_max_hp = env['team_max_hp'], env['enemy_max_hp']

        team_hp = float(team_max_hp)
        enemy_hp = float(enemy_max_hp)
        prior = utility_based_prior(player_stats, tau=tau_prior)
        turn_idx = 0
        prev_roles = None
        stages = []

        for human_combo in record['stage_roles']:
            if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                break

            stages.append({
                'prior': prior.copy(),
                'human_combo': human_combo,
                'prev_roles': prev_roles,
            })

            human_roles = [ROLE_CHAR_TO_IDX[c] for c in human_combo]
            prev_roles = list(human_roles)
            for _ in range(TURNS_PER_STAGE):
                if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                    break
                intent_t = record['lds'][turn_idx]
                actions = [preferred_action(human_roles[i], intent_t, team_hp, team_max_hp)
                           for i in range(3)]
                prior = bayesian_update(prior, actions, intent_t, team_hp, team_max_hp, epsilon)
                team_hp, enemy_hp = game_step(intent_t, team_hp, enemy_hp, actions,
                                              player_stats, boss_damage, team_max_hp)
                turn_idx += 1

        trajectories.append(stages)
    return trajectories


def predict_from_trajectory(trajectory, epsilon_switch):
    """Generate predictions from precomputed trajectory (no Bayesian updates needed)."""
    results = []
    for stage in trajectory:
        prior = stage['prior']
        prev_roles = stage['prev_roles']
        human_combo = stage['human_combo']

        switch_dist = [posterior_marginal(prior, i) for i in range(3)]

        per_agent = []
        for i in range(3):
            if prev_roles is None:
                per_agent.append(switch_dist[i])
            else:
                stick = np.zeros(3)
                stick[prev_roles[i]] = 1.0
                mixed = (1.0 - epsilon_switch) * stick + epsilon_switch * switch_dist[i]
                per_agent.append(mixed)

        predicted_dist = {}
        for r0 in range(3):
            for r1 in range(3):
                for r2 in range(3):
                    combo = ROLE_SHORT[r0] + ROLE_SHORT[r1] + ROLE_SHORT[r2]
                    predicted_dist[combo] = float(per_agent[0][r0] * per_agent[1][r1] * per_agent[2][r2])

        results.append({
            'predicted_dist': predicted_dist,
            'human_combo': human_combo,
            'model_marginal': np.mean(per_agent, axis=0),
        })
    return results

## Fine-Tuning Helpers

In [4]:
metric = 'combo_r'

def pick_best(results, metric='combo_r'):
    valid = [r for r in results if not np.isnan(r.get(metric, float('nan')))]
    if not valid:
        return results[0]
    return max(valid, key=lambda r: r[metric])


def evaluate_walk_ps(records, tau_prior, epsilon, epsilon_switch):
    """Run Bayesian Walk PS at given params (uncached, for scipy)."""
    results = run_predictions(records, make_bayesian_walk_ps(
        tau_prior=tau_prior, epsilon=epsilon, epsilon_switch=epsilon_switch))
    corrs = compute_pearson(results)
    ll = compute_log_likelihood(results)
    g = corrs.get('__global__', {})
    return {
        'tau_prior': tau_prior, 'epsilon': epsilon, 'epsilon_switch': epsilon_switch,
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }


def evaluate_from_cache(records, trajectories, epsilon_switch):
    """Evaluate using precomputed trajectories (fast)."""
    def predict_fn(record):
        idx = record['_traj_idx']
        return predict_from_trajectory(trajectories[idx], epsilon_switch)
    results = run_predictions(records, predict_fn)
    corrs = compute_pearson(results)
    ll = compute_log_likelihood(results)
    g = corrs.get('__global__', {})
    return {
        'epsilon_switch': epsilon_switch,
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }


def grid_search_walk_ps(records, tau_prior_vals, eps_vals, eps_switch_vals):
    """Cached grid search: precompute per (tau_prior, eps), sweep eps_switch."""
    for i, rec in enumerate(records):
        rec['_traj_idx'] = i

    results = []
    outer_combos = list(product(tau_prior_vals, eps_vals))
    total = len(outer_combos) * len(eps_switch_vals)
    count = 0

    for tp, eps in outer_combos:
        trajectories = precompute_trajectories(records, tp, eps)
        for eps_sw in eps_switch_vals:
            count += 1
            if count % 100 == 0 or count == total:
                print(f'  [{count}/{total}] ...', flush=True)
            res = evaluate_from_cache(records, trajectories, eps_sw)
            res['tau_prior'] = tp
            res['epsilon'] = eps
            results.append(res)
    return results

## Aggregate Fine-Tuning

In [5]:
# Grid ranges
tau_prior_min, tau_prior_max, tau_prior_steps = 0.1, 20.0, 10
eps_min, eps_max, eps_steps = 0.0, 1.0, 21
eps_switch_vals = np.linspace(0.0, 1.0, 21)

tau_prior_vals = np.linspace(tau_prior_min, tau_prior_max, tau_prior_steps)
eps_vals = np.linspace(eps_min, eps_max, eps_steps)

total = len(tau_prior_vals) * len(eps_vals) * len(eps_switch_vals)
print(f'Coarse grid: {len(tau_prior_vals)} x {len(eps_vals)} x {len(eps_switch_vals)} = {total} points')
print(f'  tau_prior:   linspace({tau_prior_min}, {tau_prior_max}, {tau_prior_steps})')
print(f'  epsilon:     linspace({eps_min}, {eps_max}, {eps_steps})')
print(f'  eps_switch:  {eps_switch_vals}')
print()

sweep_results = grid_search_walk_ps(records, tau_prior_vals, eps_vals, eps_switch_vals)
best = pick_best(sweep_results, metric)
print(f'\nCoarse best: tau_prior={best["tau_prior"]:.4f} eps={best["epsilon"]:.6f} '
      f'eps_switch={best["epsilon_switch"]:.4f}')
print(f'  combo_r={best["combo_r"]:.4f}  marg_r={best["marg_r"]:.4f}  mean_ll={best["mean_ll"]:.4f}')

Coarse grid: 10 x 21 x 21 = 4410 points
  tau_prior:   linspace(0.1, 20.0, 10)
  epsilon:     linspace(0.0, 1.0, 21)
  eps_switch:  [0.   0.05 0.1  0.15 0.2  0.25 0.3  0.35 0.4  0.45 0.5  0.55 0.6  0.65
 0.7  0.75 0.8  0.85 0.9  0.95 1.  ]

  [100/4410] ...
  [200/4410] ...


/Users/bhavyesh/Desktop/bayesian-role-specialization/analysis/shared/evaluation.py:129: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, p = pearsonr(marg_m, marg_h)


  [300/4410] ...
  [400/4410] ...
  [500/4410] ...
  [600/4410] ...
  [700/4410] ...
  [800/4410] ...
  [900/4410] ...
  [1000/4410] ...
  [1100/4410] ...
  [1200/4410] ...
  [1300/4410] ...
  [1400/4410] ...
  [1500/4410] ...
  [1600/4410] ...
  [1700/4410] ...
  [1800/4410] ...
  [1900/4410] ...
  [2000/4410] ...
  [2100/4410] ...
  [2200/4410] ...
  [2300/4410] ...
  [2400/4410] ...
  [2500/4410] ...
  [2600/4410] ...
  [2700/4410] ...
  [2800/4410] ...
  [2900/4410] ...
  [3000/4410] ...
  [3100/4410] ...
  [3200/4410] ...
  [3300/4410] ...
  [3400/4410] ...
  [3500/4410] ...
  [3600/4410] ...
  [3700/4410] ...
  [3800/4410] ...
  [3900/4410] ...
  [4000/4410] ...
  [4100/4410] ...
  [4200/4410] ...
  [4300/4410] ...
  [4400/4410] ...
  [4410/4410] ...

Coarse best: tau_prior=2.3111 eps=0.550000 eps_switch=0.5500
  combo_r=0.5248  marg_r=0.5188  mean_ll=-2.6470


In [6]:
# Refine around coarse best
tp_step = (tau_prior_max - tau_prior_min) / tau_prior_steps
eps_step_size = (eps_max - eps_min) / (eps_steps - 1)

fine_tp = np.linspace(
    max(tau_prior_min, best['tau_prior'] - tp_step),
    min(tau_prior_max, best['tau_prior'] + tp_step),
    10,
)
fine_eps = np.linspace(
    max(eps_min, best['epsilon'] - eps_step_size),
    min(eps_max, best['epsilon'] + eps_step_size),
    10,
)

def refine_range(val, vals_arr, n=8):
    sorted_vals = np.sort(vals_arr)
    idx = np.searchsorted(sorted_vals, val)
    lo = sorted_vals[max(0, idx - 1)]
    hi = sorted_vals[min(len(sorted_vals) - 1, idx + 1)] if idx < len(sorted_vals) - 1 else sorted_vals[-1]
    return np.linspace(lo, hi, n)

fine_esw = refine_range(best['epsilon_switch'], eps_switch_vals)

total_fine = len(fine_tp) * len(fine_eps) * len(fine_esw)
print(f'Refined grid: {len(fine_tp)} x {len(fine_eps)} x {len(fine_esw)} = {total_fine} points')
print(f'  tau_prior:  [{fine_tp[0]:.4f}, {fine_tp[-1]:.4f}]')
print(f'  epsilon:    [{fine_eps[0]:.6f}, {fine_eps[-1]:.6f}]')
print(f'  eps_switch: [{fine_esw[0]:.4f}, {fine_esw[-1]:.4f}]')
print()

refined_results = grid_search_walk_ps(records, fine_tp, fine_eps, fine_esw)
sweep_results.extend(refined_results)
best = pick_best(sweep_results, metric)
print(f'\nRefined best: tau_prior={best["tau_prior"]:.4f} eps={best["epsilon"]:.6f} '
      f'eps_switch={best["epsilon_switch"]:.4f}')
print(f'  combo_r={best["combo_r"]:.4f}  marg_r={best["marg_r"]:.4f}  mean_ll={best["mean_ll"]:.4f}')

Refined grid: 10 x 10 x 8 = 800 points
  tau_prior:  [0.3211, 4.3011]
  epsilon:    [0.500000, 0.600000]
  eps_switch: [0.5000, 0.6000]

  [100/800] ...
  [200/800] ...
  [300/800] ...
  [400/800] ...
  [500/800] ...
  [600/800] ...
  [700/800] ...
  [800/800] ...

Refined best: tau_prior=1.6478 eps=0.500000 eps_switch=0.5000
  combo_r=0.5265  marg_r=0.5183  mean_ll=-2.8281


In [7]:
# Scipy L-BFGS-B polishing
def objective(params):
    tp, eps, eps_sw = params
    res = evaluate_walk_ps(records, tp, eps, eps_sw)
    return -res[metric]

tp_lo = max(tau_prior_min, best['tau_prior'] - tp_step / 2)
tp_hi = min(tau_prior_max, best['tau_prior'] + tp_step / 2)
eps_lo = max(eps_min, best['epsilon'] - eps_step_size / 2)
eps_hi = min(eps_max, best['epsilon'] + eps_step_size / 2)
esw_lo = max(0.01, best['epsilon_switch'] - 0.1)
esw_hi = min(1.0, best['epsilon_switch'] + 0.1)

x0 = [best['tau_prior'], best['epsilon'], best['epsilon_switch']]
bounds = [(tp_lo, tp_hi), (eps_lo, eps_hi), (esw_lo, esw_hi)]

print(f'Scipy optimization (metric={metric}) ...')
print(f'  bounds: tau_prior=[{tp_lo:.4f}, {tp_hi:.4f}]')
print(f'          epsilon=[{eps_lo:.6f}, {eps_hi:.6f}], eps_switch=[{esw_lo:.4f}, {esw_hi:.4f}]')

opt_result = minimize(objective, x0, method='L-BFGS-B', bounds=bounds,
                      options={'maxiter': 50, 'ftol': 1e-6})
opt_tp, opt_eps, opt_esw = opt_result.x
print(f'  Optimal: tau_prior={opt_tp:.4f} eps={opt_eps:.6f} eps_switch={opt_esw:.4f}')
print(f'  objective={-opt_result.fun:.4f}')

opt_eval = evaluate_walk_ps(records, opt_tp, opt_eps, opt_esw)
sweep_results.append(opt_eval)
best = pick_best(sweep_results, metric)
print(f'\nFinal best: tau_prior={best["tau_prior"]:.4f} eps={best["epsilon"]:.6f} '
      f'eps_switch={best["epsilon_switch"]:.4f}')
print(f'  combo_r={best["combo_r"]:.4f}  marg_r={best["marg_r"]:.4f}  mean_ll={best["mean_ll"]:.4f}')

Scipy optimization (metric=combo_r) ...
  bounds: tau_prior=[0.6528, 2.6428]
          epsilon=[0.475000, 0.525000], eps_switch=[0.4000, 0.6000]
  Optimal: tau_prior=1.7412 eps=0.489074 eps_switch=0.4760
  objective=0.5272

Final best: tau_prior=1.7412 eps=0.489074 eps_switch=0.4760
  combo_r=0.5272  marg_r=0.5223  mean_ll=-2.8532


## Switch-Stage Fine-Tuning

In [8]:
def filter_switch_stages(all_results):
    """Keep only predictions where the human combo changed from the previous stage."""
    filtered = {}
    for env_id, data in all_results.items():
        new_team_preds = []
        for team_preds in data['team_predictions']:
            filtered_preds = []
            for s, pred in enumerate(team_preds):
                if s > 0 and pred['human_combo'] != team_preds[s - 1]['human_combo']:
                    filtered_preds.append(pred)
            new_team_preds.append(filtered_preds)

        canon_combos = data['canonical_combos']
        stat_profile = data['stat_profile']
        stage_predicted = defaultdict(lambda: defaultdict(float))
        stage_human = defaultdict(lambda: defaultdict(int))
        stage_model_marg = defaultdict(lambda: np.zeros(3))
        stage_human_marg = defaultdict(lambda: np.zeros(3))
        stage_counts = defaultdict(int)
        max_stages = 0

        for team_preds in new_team_preds:
            for s, pred in enumerate(team_preds):
                stage_counts[s] += 1
                max_stages = max(max_stages, s + 1)
                for combo, prob in pred['predicted_dist'].items():
                    stage_predicted[s][canonical_combo(combo, stat_profile)] += prob
                stage_human[s][canonical_combo(pred['human_combo'], stat_profile)] += 1
                stage_model_marg[s] += pred['model_marginal']
                stage_human_marg[s] += combo_marginal(pred['human_combo'])

        if max_stages == 0:
            continue

        predicted_avg, model_marg_avg, human_marg_avg = {}, {}, {}
        for s in range(max_stages):
            n = stage_counts[s]
            if n > 0:
                predicted_avg[s] = {cc: stage_predicted[s].get(cc, 0.0) / n for cc in canon_combos}
                model_marg_avg[s] = stage_model_marg[s] / n
                human_marg_avg[s] = stage_human_marg[s] / n

        filtered[env_id] = dict(data)
        filtered[env_id].update({
            'max_stages': max_stages,
            'stage_predicted': predicted_avg,
            'stage_human': dict(stage_human),
            'stage_counts': dict(stage_counts),
            'team_predictions': new_team_preds,
            'stage_model_marginal': model_marg_avg,
            'stage_human_marginal': human_marg_avg,
        })

    return filtered


def evaluate_walk_ps_switch(records, tau_prior, epsilon, epsilon_switch):
    """Run on full history, compute metrics only on switch stages."""
    full_results = run_predictions(records, make_bayesian_walk_ps(
        tau_prior=tau_prior, epsilon=epsilon, epsilon_switch=epsilon_switch))
    sw_results = filter_switch_stages(full_results)
    if not sw_results:
        return {'tau_prior': tau_prior, 'epsilon': epsilon, 'epsilon_switch': epsilon_switch,
                'combo_r': float('nan'), 'marg_r': float('nan'), 'mean_ll': float('nan')}
    corrs = compute_pearson(sw_results)
    ll = compute_log_likelihood(sw_results)
    g = corrs.get('__global__', {})
    return {
        'tau_prior': tau_prior, 'epsilon': epsilon, 'epsilon_switch': epsilon_switch,
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }


def evaluate_switch_from_cache(records, trajectories, epsilon_switch):
    """Evaluate switch-stage metrics using precomputed trajectories."""
    def predict_fn(record):
        idx = record['_traj_idx']
        return predict_from_trajectory(trajectories[idx], epsilon_switch)
    full_results = run_predictions(records, predict_fn)
    sw_results = filter_switch_stages(full_results)
    if not sw_results:
        return {'epsilon_switch': epsilon_switch,
                'combo_r': float('nan'), 'marg_r': float('nan'), 'mean_ll': float('nan')}
    corrs = compute_pearson(sw_results)
    ll = compute_log_likelihood(sw_results)
    g = corrs.get('__global__', {})
    return {
        'epsilon_switch': epsilon_switch,
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }


def grid_search_walk_ps_switch(records, tau_prior_vals, eps_vals, eps_switch_vals):
    """Cached grid search optimizing switch-stage correlation."""
    for i, rec in enumerate(records):
        rec['_traj_idx'] = i

    results = []
    outer_combos = list(product(tau_prior_vals, eps_vals))
    total = len(outer_combos) * len(eps_switch_vals)
    count = 0

    for tp, eps in outer_combos:
        trajectories = precompute_trajectories(records, tp, eps)
        for eps_sw in eps_switch_vals:
            count += 1
            if count % 100 == 0 or count == total:
                print(f'  [{count}/{total}] ...', flush=True)
            res = evaluate_switch_from_cache(records, trajectories, eps_sw)
            res['tau_prior'] = tp
            res['epsilon'] = eps
            results.append(res)
    return results

In [9]:
# Coarse grid (same ranges)
print('=== Switch-Stage Fine-Tuning: Coarse Grid ===')
sw_total = len(tau_prior_vals) * len(eps_vals) * len(eps_switch_vals)
print(f'{sw_total} points\n')

sw_sweep = grid_search_walk_ps_switch(records, tau_prior_vals, eps_vals, eps_switch_vals)
best_sw = pick_best(sw_sweep, metric)
print(f'\nCoarse best: tau_prior={best_sw["tau_prior"]:.4f} eps={best_sw["epsilon"]:.6f} '
      f'eps_switch={best_sw["epsilon_switch"]:.4f}')
print(f'  combo_r={best_sw["combo_r"]:.4f}  marg_r={best_sw["marg_r"]:.4f}  mean_ll={best_sw["mean_ll"]:.4f}')

=== Switch-Stage Fine-Tuning: Coarse Grid ===
4410 points

  [100/4410] ...
  [200/4410] ...
  [300/4410] ...
  [400/4410] ...
  [500/4410] ...
  [600/4410] ...
  [700/4410] ...
  [800/4410] ...
  [900/4410] ...
  [1000/4410] ...
  [1100/4410] ...
  [1200/4410] ...
  [1300/4410] ...
  [1400/4410] ...
  [1500/4410] ...
  [1600/4410] ...
  [1700/4410] ...
  [1800/4410] ...
  [1900/4410] ...
  [2000/4410] ...
  [2100/4410] ...
  [2200/4410] ...
  [2300/4410] ...
  [2400/4410] ...
  [2500/4410] ...
  [2600/4410] ...
  [2700/4410] ...
  [2800/4410] ...
  [2900/4410] ...
  [3000/4410] ...
  [3100/4410] ...
  [3200/4410] ...
  [3300/4410] ...
  [3400/4410] ...
  [3500/4410] ...
  [3600/4410] ...
  [3700/4410] ...
  [3800/4410] ...
  [3900/4410] ...
  [4000/4410] ...
  [4100/4410] ...
  [4200/4410] ...
  [4300/4410] ...
  [4400/4410] ...
  [4410/4410] ...

Coarse best: tau_prior=20.0000 eps=0.500000 eps_switch=1.0000
  combo_r=0.4306  marg_r=0.5667  mean_ll=-3.0287


In [10]:
# Refined grid around switch-stage coarse best
fine_sw_tp = np.linspace(
    max(tau_prior_min, best_sw['tau_prior'] - tp_step),
    min(tau_prior_max, best_sw['tau_prior'] + tp_step),
    10,
)
fine_sw_eps = np.linspace(
    max(eps_min, best_sw['epsilon'] - eps_step_size),
    min(eps_max, best_sw['epsilon'] + eps_step_size),
    10,
)
fine_sw_esw = refine_range(best_sw['epsilon_switch'], eps_switch_vals)

total_sw_fine = len(fine_sw_tp) * len(fine_sw_eps) * len(fine_sw_esw)
print(f'Refined grid (switch-stage): {total_sw_fine} points')
print(f'  tau_prior:  [{fine_sw_tp[0]:.4f}, {fine_sw_tp[-1]:.4f}]')
print(f'  epsilon:    [{fine_sw_eps[0]:.6f}, {fine_sw_eps[-1]:.6f}]')
print(f'  eps_switch: [{fine_sw_esw[0]:.4f}, {fine_sw_esw[-1]:.4f}]')
print()

sw_refined = grid_search_walk_ps_switch(records, fine_sw_tp, fine_sw_eps, fine_sw_esw)
sw_sweep.extend(sw_refined)
best_sw = pick_best(sw_sweep, metric)
print(f'\nRefined best: tau_prior={best_sw["tau_prior"]:.4f} eps={best_sw["epsilon"]:.6f} '
      f'eps_switch={best_sw["epsilon_switch"]:.4f}')
print(f'  combo_r={best_sw["combo_r"]:.4f}  marg_r={best_sw["marg_r"]:.4f}  mean_ll={best_sw["mean_ll"]:.4f}')

Refined grid (switch-stage): 800 points
  tau_prior:  [18.0100, 20.0000]
  epsilon:    [0.450000, 0.550000]
  eps_switch: [0.9500, 1.0000]

  [100/800] ...
  [200/800] ...
  [300/800] ...
  [400/800] ...
  [500/800] ...
  [600/800] ...
  [700/800] ...
  [800/800] ...

Refined best: tau_prior=20.0000 eps=0.483333 eps_switch=1.0000
  combo_r=0.4308  marg_r=0.5614  mean_ll=-3.0834


In [11]:
# Scipy polishing for switch-stage tuning
def objective_sw(params):
    tp, eps, eps_sw = params
    res = evaluate_walk_ps_switch(records, tp, eps, eps_sw)
    return -res[metric]

sw_tp_lo = max(tau_prior_min, best_sw['tau_prior'] - tp_step / 2)
sw_tp_hi = min(tau_prior_max, best_sw['tau_prior'] + tp_step / 2)
sw_eps_lo = max(eps_min, best_sw['epsilon'] - eps_step_size / 2)
sw_eps_hi = min(eps_max, best_sw['epsilon'] + eps_step_size / 2)
sw_esw_lo = max(0.01, best_sw['epsilon_switch'] - 0.1)
sw_esw_hi = min(1.0, best_sw['epsilon_switch'] + 0.1)

sw_x0 = [best_sw['tau_prior'], best_sw['epsilon'], best_sw['epsilon_switch']]
sw_bounds = [(sw_tp_lo, sw_tp_hi), (sw_eps_lo, sw_eps_hi), (sw_esw_lo, sw_esw_hi)]

print('Scipy optimization (switch-stage) ...')
print(f'  bounds: tau_prior=[{sw_tp_lo:.4f}, {sw_tp_hi:.4f}]')
print(f'          epsilon=[{sw_eps_lo:.6f}, {sw_eps_hi:.6f}], eps_switch=[{sw_esw_lo:.4f}, {sw_esw_hi:.4f}]')

sw_opt = minimize(objective_sw, sw_x0, method='L-BFGS-B', bounds=sw_bounds,
                  options={'maxiter': 50, 'ftol': 1e-6})
sw_opt_tp, sw_opt_eps, sw_opt_esw = sw_opt.x
print(f'  Optimal: tau_prior={sw_opt_tp:.4f} eps={sw_opt_eps:.6f} eps_switch={sw_opt_esw:.4f}')
print(f'  objective={-sw_opt.fun:.4f}')

sw_opt_eval = evaluate_walk_ps_switch(records, sw_opt_tp, sw_opt_eps, sw_opt_esw)
sw_sweep.append(sw_opt_eval)
best_sw = pick_best(sw_sweep, metric)
print(f'\nFinal best (switch-stage): tau_prior={best_sw["tau_prior"]:.4f} '
      f'eps={best_sw["epsilon"]:.6f} eps_switch={best_sw["epsilon_switch"]:.4f}')
print(f'  combo_r={best_sw["combo_r"]:.4f}  marg_r={best_sw["marg_r"]:.4f}  mean_ll={best_sw["mean_ll"]:.4f}')

Scipy optimization (switch-stage) ...
  bounds: tau_prior=[19.0050, 20.0000]
          epsilon=[0.458333, 0.508333], eps_switch=[0.9000, 1.0000]
  Optimal: tau_prior=20.0000 eps=0.486602 eps_switch=1.0000
  objective=0.4308

Final best (switch-stage): tau_prior=20.0000 eps=0.486602 eps_switch=1.0000
  combo_r=0.4308  marg_r=0.5624  mean_ll=-3.0720


## Save Parameters

In [12]:
output = {
    'metric_optimized': metric,
    'aggregate_tuned': {
        'tau_prior': best['tau_prior'], 'epsilon': best['epsilon'],
        'epsilon_switch': best['epsilon_switch'],
        'combo_r': best['combo_r'], 'marg_r': best['marg_r'], 'mean_ll': best['mean_ll'],
    },
    'switch_stage_tuned': {
        'tau_prior': best_sw['tau_prior'], 'epsilon': best_sw['epsilon'],
        'epsilon_switch': best_sw['epsilon_switch'],
        'combo_r': best_sw['combo_r'], 'marg_r': best_sw['marg_r'], 'mean_ll': best_sw['mean_ll'],
    },
}

with open('bayesian_walk_ps_params.json', 'w') as f:
    json.dump(output, f, indent=2)
print('Saved to bayesian_walk_ps_params.json')
print(json.dumps(output, indent=2))

Saved to bayesian_walk_ps_params.json
{
  "metric_optimized": "combo_r",
  "aggregate_tuned": {
    "tau_prior": 1.7411669435386208,
    "epsilon": 0.48907369923707505,
    "epsilon_switch": 0.4760040431270125,
    "combo_r": 0.5272311449212697,
    "marg_r": 0.5222597359952852,
    "mean_ll": -2.853214219268155
  },
  "switch_stage_tuned": {
    "tau_prior": 20.0,
    "epsilon": 0.4866017196491444,
    "epsilon_switch": 1.0,
    "combo_r": 0.4308031448634344,
    "marg_r": 0.5624062698272309,
    "mean_ll": -3.0720064552721227
  }
}
